# Full-release I-HAZE / NH-HAZE evaluation (Kaggle, inference only)

Evaluate the three fixed ECA-LDNet Stage 4 checkpoints on all released paired I-HAZE (35 pairs) and NH-HAZE (55 pairs), with PSNR, SSIM, MS-SSIM, MAE, CIEDE2000, LPIPS, and DISTS. The model weights, ASM blend (0.08), and epsilon (0.1) stay fixed. No training, tuning, or per-image selection occurs.

**Important split reporting:** the full releases include training, validation, and test images. This notebook keeps those partitions separate when their directory names are available. Report the official `test` split separately. Results pooled across `train`/`validation`/`test` are descriptive all-release transfer measurements, not official test-set scores or challenge results. Do not use them to select settings.

The prior Kaggle inputs used in this project mounted only five test pairs per dataset. Attach the **complete released datasets**; setup checks for 35 and 55 matched pairs and stops if it finds only the five-pair test subset. The official dataset pages describe the releases: [I-HAZE](https://data.vision.ee.ethz.ch/cvl/ntire18/i-haze/) and [NH-HAZE](https://data.vision.ee.ethz.ch/cvl/ntire20/nh-haze/).

## Kaggle setup

1. Add the reviewer evaluation package containing `code/scripts/evaluate_paired.py`, `code/eca_ldnet/`, and the three full-model Stage 4 checkpoints.
2. Add complete I-HAZE and NH-HAZE releases as Kaggle inputs. Edit the input paths in the configuration cell if their mount paths differ.
3. Enable a GPU and Internet. Internet is used to install the optional IQA package and fetch its pretrained LPIPS/DISTS weights.
4. Run cells in order. Setup, data scanning, each dataset/seed inference run, summaries, plots, and ZIP export are separate cells. Inference streams per-image log lines into the running cell output so a slow task can be identified.

The full-release result improves sample coverage but does not validate architectural novelty or isolate component effects. The all-release summary includes images that are not official test data.

## 1. Imports, editable paths, and run folder

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import platform
import re
import subprocess
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

INPUT_ROOT = Path("/kaggle/input")
PACKAGE_HINT = Path("/kaggle/input/datasets/randomguy28/reviewer-test-packge/reviewer_test_package")
DATASET_ROOTS = {
    "I-HAZE": Path("/kaggle/input/datasets/litheshshetty/i-haze/I-HAZE"),
    "NH-HAZE": Path("/kaggle/input/datasets/litheshshetty/nh-haze/NH-HAZE"),
}
OUTPUT_BASE = Path("/kaggle/working/full_release_real_haze_perceptual_eval")
EXPECTED_TOTALS = {"I-HAZE": 35, "NH-HAZE": 55}
SEEDS = (17, 43, 73)
BOOTSTRAP_SAMPLES = 10_000
BOOTSTRAP_SEED = 20261014
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
METRICS = ("psnr", "ssim", "ms_ssim", "mae", "deltaE2000", "lpips", "dists")

RUN_ID = time.strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = OUTPUT_BASE / f"run_{RUN_ID}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Run ID:", RUN_ID)
print("Output:", OUTPUT_ROOT)

## 2. Perceptual-metric dependencies

In [ ]:
import torch

torch_version_before = torch.__version__
if importlib.util.find_spec("pyiqa") is None:
    print("Installing IQA-PyTorch and its small dependencies…", flush=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
        "pyiqa==0.1.13", "addict", "einops", "future", "ftfy", "timm==0.9.16", "yapf"
    ], check=True)
else:
    print("pyiqa is already installed.", flush=True)
import pyiqa
assert torch.__version__ == torch_version_before, "Torch changed during IQA setup; restart with Kaggle's original runtime."
print("Torch:", torch.__version__, "pyiqa:", pyiqa.__version__)

## 3. TensorFlow and GPU initialization

In [ ]:
import tensorflow as tf

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
print("TensorFlow:", tf.__version__)
print("TensorFlow GPUs:", [d.name for d in tf.config.list_physical_devices("GPU")])

## 4. Locate evaluator and fixed checkpoints

In [ ]:
def find_evaluator_script():
    preferred = PACKAGE_HINT / "code" / "scripts" / "evaluate_paired.py"
    if preferred.is_file():
        return preferred
    matches = sorted(INPUT_ROOT.rglob("evaluate_paired.py"))
    matches = [p for p in matches if p.parent.name == "scripts" and
               (p.parents[1] / "eca_ldnet" / "model.py").is_file()]
    if not matches:
        raise FileNotFoundError("Attach the reviewer package with code/scripts/evaluate_paired.py and code/eca_ldnet/.")
    return matches[0]

EVALUATOR_SCRIPT = find_evaluator_script()
CODE_ROOT = EVALUATOR_SCRIPT.parents[1]
PACKAGE_ROOT = CODE_ROOT.parent
sys.path.insert(0, str(CODE_ROOT))
from eca_ldnet.data import get_gt_path

def find_checkpoint(seed):
    preferred = PACKAGE_ROOT / "models" / "full" / f"seed{seed}" / "stage4_best.keras"
    if preferred.is_file():
        return preferred
    candidates = sorted(INPUT_ROOT.rglob("stage4_best.keras"))
    candidates = [p for p in candidates if f"seed{seed}" in {part.lower() for part in p.parts}
                  and "full" in {part.lower() for part in p.parts}]
    if not candidates:
        raise FileNotFoundError(f"No full-model Stage 4 best checkpoint for seed {seed} under {INPUT_ROOT}.")
    return candidates[0]

CHECKPOINTS = {seed: find_checkpoint(seed) for seed in SEEDS}
print("Evaluator:", EVALUATOR_SCRIPT)
print("Package:", PACKAGE_ROOT)
for seed, checkpoint in CHECKPOINTS.items():
    print(f"Seed {seed} checkpoint: {checkpoint}")

## 5. Define paired-data layout scanner

In [ ]:
GT_DIR_NAMES = {"gt", "groundtruth", "ground_truth", "clear", "clean", "haze-free", "haze_free"}
HAZY_DIR_NAMES = {"hazy", "haze", "hazy_images", "hazyimages"}
SPLIT_NAMES = {
    "train": "train", "training": "train",
    "val": "validation", "valid": "validation", "validation": "validation",
    "test": "test", "testing": "test",
}

def image_files(folder):
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS)

def split_label(root, hazy_dir):
    parts = [p.lower().replace(" ", "_") for p in hazy_dir.relative_to(root).parts[:-1]]
    labels = [SPLIT_NAMES[p] for p in parts if p in SPLIT_NAMES]
    return labels[-1] if labels else "all_released_unpartitioned"

def discover_layouts(dataset_name, root):
    if not root.is_dir():
        raise FileNotFoundError(f"{dataset_name} root is missing: {root}. Edit DATASET_ROOTS and attach the complete release.")
    layouts, seen_hazy = [], set()
    for current, dirs, _ in os.walk(root):
        dirs[:] = [d for d in dirs if not d.startswith(".")]
        current_path = Path(current)
        if current_path.name.lower() not in HAZY_DIR_NAMES:
            continue
        hazy_dir = current_path
        sibling_dirs = [p for p in hazy_dir.parent.iterdir() if p.is_dir()]
        gt_candidates = [p for p in sibling_dirs if p.name.lower() in GT_DIR_NAMES]
        if not gt_candidates:
            continue
        gt_dir = sorted(gt_candidates, key=lambda p: p.name.lower())[0]
        hazy_files = image_files(hazy_dir)
        pairs = [(f, get_gt_path(f.name, str(gt_dir))) for f in hazy_files]
        pairs = [(f, Path(g)) for f, g in pairs if g]
        key = (str(hazy_dir.resolve()), str(gt_dir.resolve()))
        if not pairs or key in seen_hazy:
            continue
        seen_hazy.add(key)
        layouts.append({"dataset": dataset_name, "split": split_label(root, hazy_dir),
                        "hazy_dir": hazy_dir, "gt_dir": gt_dir,
                        "pairs": pairs, "n_pairs": len(pairs)})
    if not layouts:
        raise RuntimeError(f"No paired hazy/GT layout found under {root}; expected sibling hazy and GT/clear folders.")
    return layouts



## 6. Scan and validate full releases

In [ ]:
LAYOUTS = {}
for name, root in DATASET_ROOTS.items():
    print(f"Scanning {name}: {root}", flush=True)
    LAYOUTS[name] = discover_layouts(name, root)
    print(f"Found layouts: {[(x['split'], x['n_pairs']) for x in LAYOUTS[name]]}", flush=True)

PAIR_INVENTORY = []
for dataset_name, layouts in LAYOUTS.items():
    total = sum(item["n_pairs"] for item in layouts)
    print(dataset_name, "total pairs:", total,
          "by split:", {item["split"]: item["n_pairs"] for item in layouts})
    expected = EXPECTED_TOTALS[dataset_name]
    if total != expected:
        raise RuntimeError(
            f"{dataset_name}: found {total} matched pairs; expected all {expected} released pairs. "
            "The attached input may contain only the five-image test split. Attach the complete release and update DATASET_ROOTS."
        )
    checked = 0
    for item in layouts:
        for hazy_path, gt_path in item["pairs"]:
            hazy = cv2.imread(str(hazy_path), cv2.IMREAD_COLOR)
            gt = cv2.imread(str(gt_path), cv2.IMREAD_COLOR)
            if hazy is None or gt is None:
                raise RuntimeError(f"Unreadable pair: {hazy_path} / {gt_path}")
            PAIR_INVENTORY.append({
                "dataset": dataset_name, "split": item["split"], "image": hazy_path.name,
                "hazy_path": str(hazy_path), "gt_path": str(gt_path),
                "hazy_height": int(hazy.shape[0]), "hazy_width": int(hazy.shape[1]),
                "gt_height": int(gt.shape[0]), "gt_width": int(gt.shape[1]),
            })
            checked += 1
            if checked % 10 == 0 or checked == total:
                print(f"  validated {dataset_name}: {checked}/{total} pairs", flush=True)

inventory = pd.DataFrame(PAIR_INVENTORY)
if (inventory[["hazy_height", "hazy_width"]].to_numpy() != inventory[["gt_height", "gt_width"]].to_numpy()).any():
    raise RuntimeError("Some hazy/GT image dimensions differ. Resolve alignment before inference.")
if inventory.duplicated(["dataset", "split", "image"]).any():
    raise RuntimeError("Duplicate dataset/split/image keys found; fix the input directory structure.")
inventory.to_csv(OUTPUT_ROOT / "pair_inventory.csv", index=False)
print("Total matched pairs:", len(inventory))

## 7. Define one resumable, streaming evaluation task

Each following dataset/seed cell runs only that checkpoint. Its log streams one line at a time and is also saved to `evaluation.log`. If a cell is interrupted, rerunning that same cell reuses the partial output directory; completed tasks are skipped.

In [ ]:
RUN_DIRS = {}

def run_one_layout_seed(dataset_name, layout, seed):
    split = layout["split"]
    dataset_label = f"{dataset_name}_{split}"
    out_dir = OUTPUT_ROOT / "predictions" / dataset_name / split / f"seed{seed}"
    detail = out_dir / "paired_per_image.csv"
    manifest = out_dir / "paired_run_manifest.json"
    key = (dataset_name, split, seed)
    if detail.is_file() and manifest.is_file():
        n_rows = len(pd.read_csv(detail))
        if n_rows == layout["n_pairs"]:
            RUN_DIRS[key] = out_dir
            print(f"[SKIP completed] {dataset_label}, seed {seed}: {n_rows} pairs", flush=True)
            return

    out_dir.mkdir(parents=True, exist_ok=True)
    log_path = out_dir / "evaluation.log"
    cmd = [
        sys.executable, "-u", str(EVALUATOR_SCRIPT),
        "--model", str(CHECKPOINTS[seed]), "--variant", "full",
        "--hazy-dir", str(layout["hazy_dir"]), "--gt-dir", str(layout["gt_dir"]),
        "--dataset", dataset_label, "--output", str(out_dir),
        "--preprocessing", "radiometric", "--native-resolution",
        "--perceptual", "lpips", "dists",
        "--bootstrap-samples", str(BOOTSTRAP_SAMPLES), "--seed", str(BOOTSTRAP_SEED),
    ]
    print(f"[START] {dataset_label}, seed {seed}, n={layout['n_pairs']}", flush=True)
    print("Log:", log_path, flush=True)
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end="", flush=True)
        return_code = process.wait()
    if return_code:
        tail = log_path.read_text(encoding="utf-8", errors="replace")[-8000:]
        raise RuntimeError(f"Evaluation failed for {dataset_label}, seed {seed}. Log tail:\n{tail}")
    if not detail.is_file() or not manifest.is_file():
        raise RuntimeError(f"Expected evaluator output missing under {out_dir}")
    n_rows = len(pd.read_csv(detail))
    if n_rows != layout["n_pairs"]:
        raise RuntimeError(f"Expected {layout['n_pairs']} rows for {dataset_label}, seed {seed}; found {n_rows}")
    RUN_DIRS[key] = out_dir
    print(f"[DONE] {dataset_label}, seed {seed}: {n_rows} pairs in {(time.perf_counter()-started)/60:.1f} min", flush=True)

def run_dataset_seed(dataset_name, seed):
    layouts = LAYOUTS[dataset_name]
    print(f"=== {dataset_name}, seed {seed}: {len(layouts)} split(s) ===", flush=True)
    for layout in layouts:
        run_one_layout_seed(dataset_name, layout, seed)
    print(f"=== Finished {dataset_name}, seed {seed} ===", flush=True)

## 8. Inference runs — execute each cell separately

### I-HAZE — seed 17

In [ ]:
run_dataset_seed("I-HAZE", 17)

### I-HAZE — seed 43

In [ ]:
run_dataset_seed("I-HAZE", 43)

### I-HAZE — seed 73

In [ ]:
run_dataset_seed("I-HAZE", 73)

### NH-HAZE — seed 17

In [ ]:
run_dataset_seed("NH-HAZE", 17)

### NH-HAZE — seed 43

In [ ]:
run_dataset_seed("NH-HAZE", 43)

### NH-HAZE — seed 73

In [ ]:
run_dataset_seed("NH-HAZE", 73)

## 9. Confirm all inference tasks finished

In [ ]:
expected_tasks = sum(len(layouts) for layouts in LAYOUTS.values()) * len(SEEDS)
if len(RUN_DIRS) != expected_tasks:
    missing = [
        (name, layout["split"], seed)
        for name, layouts in LAYOUTS.items()
        for layout in layouts
        for seed in SEEDS
        if (name, layout["split"], seed) not in RUN_DIRS
    ]
    raise RuntimeError(f"Inference is incomplete: {len(RUN_DIRS)}/{expected_tasks} tasks. Missing: {missing}")
print(f"All {expected_tasks} dataset/split/checkpoint tasks are complete.")

## 10. Load per-image outputs and verify pair counts

In [ ]:
frames = []
for (dataset_name, split, seed), out_dir in RUN_DIRS.items():
    frame = pd.read_csv(out_dir / "paired_per_image.csv")
    if len(frame) != next(x["n_pairs"] for x in LAYOUTS[dataset_name] if x["split"] == split):
        raise RuntimeError(f"Pair count changed for {dataset_name}/{split}/seed{seed}")
    frame["dataset_name"] = dataset_name
    frame["split"] = split
    frame["seed"] = seed
    frames.append(frame)

scores = pd.concat(frames, ignore_index=True)
score_path = OUTPUT_ROOT / "per_image_all_seeds.csv"
scores.to_csv(score_path, index=False)
metric_cols = [m for m in METRICS if m in scores.columns]
expected_rows = len(inventory) * len(SEEDS)
if len(scores) != expected_rows:
    raise RuntimeError(f"Expected {expected_rows} per-seed rows, found {len(scores)}")



## 11. Build seed, cross-seed, and per-image summaries

In [ ]:
metric_cols = [m for m in METRICS if m in scores.columns]
expected_rows = len(inventory) * len(SEEDS)
if len(scores) != expected_rows:
    raise RuntimeError(f"Expected {expected_rows} per-seed rows, found {len(scores)}")

seed_rows = []
for (dataset_name, split, seed), group in scores.groupby(["dataset_name", "split", "seed"], sort=True):
    for metric in metric_cols:
        values = group[metric].to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        seed_rows.append({"dataset": dataset_name, "split": split, "seed": seed,
                          "metric": metric, "n_images": len(values),
                          "mean": float(np.mean(values)),
                          "image_sd": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
                          "median": float(np.median(values))})
seed_summary = pd.DataFrame(seed_rows)
seed_summary.to_csv(OUTPUT_ROOT / "seed_level_summary.csv", index=False)

cross_seed = (seed_summary.groupby(["dataset", "split", "metric"], as_index=False)
              .agg(n_seeds=("seed", "nunique"), mean_of_seed_means=("mean", "mean"),
                   sd_across_seed_means=("mean", "std")))
cross_seed.to_csv(OUTPUT_ROOT / "cross_seed_summary.csv", index=False)

image_means = (scores.groupby(["dataset_name", "split", "image"], as_index=False)[metric_cols]
               .mean())
image_means.to_csv(OUTPUT_ROOT / "per_image_seed_mean.csv", index=False)


## 12. Bootstrap image-clustered intervals

In [ ]:
rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_rows = []
for (dataset_name, split), group in image_means.groupby(["dataset_name", "split"], sort=True):
    for metric in metric_cols:
        values = group[metric].to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        if not len(values):
            continue
        draws = rng.integers(0, len(values), size=(BOOTSTRAP_SAMPLES, len(values)))
        means = values[draws].mean(axis=1)
        bootstrap_rows.append({
            "dataset": dataset_name, "split": split, "metric": metric, "n_images": len(values),
            "mean_across_images_and_seeds": float(values.mean()),
            "bootstrap_ci95_low": float(np.quantile(means, 0.025)),
            "bootstrap_ci95_high": float(np.quantile(means, 0.975)),
            "bootstrap_unit": "image; all three fixed seed outcomes retained",
        })
bootstrap_summary = pd.DataFrame(bootstrap_rows)
bootstrap_summary.to_csv(OUTPUT_ROOT / "image_cluster_bootstrap_summary.csv", index=False)
print("Cross-seed summary:")
print(cross_seed.to_string(index=False))
print("Image-clustered intervals:")
print(bootstrap_summary.to_string(index=False))

## 13. Plot metric distributions

In [ ]:
# Distribution plots: one plot per dataset/split, with paired images shown as per-seed boxplots.
for (dataset_name, split), group in scores.groupby(["dataset_name", "split"], sort=True):
    plot_metrics = [m for m in ("psnr", "lpips", "dists") if m in group.columns]
    fig, axes = plt.subplots(1, len(plot_metrics), figsize=(5.2 * len(plot_metrics), 4.2), squeeze=False)
    for ax, metric in zip(axes[0], plot_metrics):
        data = [group.loc[group["seed"] == seed, metric].dropna().to_numpy() for seed in SEEDS]
        ax.boxplot(data, tick_labels=[f"seed {seed}" for seed in SEEDS], showfliers=True)
        ax.set_title(metric.upper() + (" (higher better)" if metric == "psnr" else " (lower better)"))
        ax.grid(axis="y", alpha=0.25)
    fig.suptitle(f"{dataset_name} — {split}: all paired images")
    fig.tight_layout()
    fig.savefig(OUTPUT_ROOT / f"distribution_{dataset_name}_{split}.png", dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)



## 14. Generate ranked-case atlases

In [ ]:
case_rows = []
for dataset_name, layouts in LAYOUTS.items():
    for layout in layouts:
        split = layout["split"]
        group = image_means[(image_means["dataset_name"] == dataset_name) & (image_means["split"] == split)]
        ranked = group.sort_values("psnr", kind="stable").reset_index(drop=True)
        n = len(ranked)
        positions = [0, round(0.25 * (n - 1)), round(0.50 * (n - 1)), n - 1]
        labels = ["lowest", "lower_quartile", "median", "highest"]
        chosen = []
        for label, position in zip(labels, positions):
            row = ranked.iloc[position]
            if row["image"] not in [item["image"] for item in chosen]:
                record = {"dataset": dataset_name, "split": split, "selection": label,
                          "image": row["image"], "mean_psnr_across_seeds": float(row["psnr"])}
                chosen.append(record)
                case_rows.append(record)
        pair_map = {hazy.name: (hazy, gt) for hazy, gt in layout["pairs"]}
        safe_label = re.sub(r"[^A-Za-z0-9._-]+", "_", f"{dataset_name}_{split}").strip("_")
        fig, axes = plt.subplots(len(chosen), 5, figsize=(17, 3.6 * len(chosen)), squeeze=False)
        col_titles = ["Hazy", "Ground truth", "Seed 17", "Seed 43", "Seed 73"]
        for col, title in enumerate(col_titles):
            axes[0, col].set_title(title, fontsize=11)
        for row_index, case in enumerate(chosen):
            hazy_path, gt_path = pair_map[case["image"]]
            paths = [hazy_path, gt_path]
            for seed in SEEDS:
                sample_dir = RUN_DIRS[(dataset_name, split, seed)] / f"tested_sample_{safe_label}"
                paths.append(sample_dir / (hazy_path.stem + ".png"))
            for col, image_path in enumerate(paths):
                image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
                if image is None:
                    raise FileNotFoundError(f"Expected atlas image is missing: {image_path}")
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                axes[row_index, col].imshow(image)
                axes[row_index, col].axis("off")
            axes[row_index, 0].set_ylabel(
                f"{case['selection']}\n{case['image']}\nmean PSNR={case['mean_psnr_across_seeds']:.2f} dB",
                fontsize=9,
            )
        fig.suptitle(f"{dataset_name} — {split}: PSNR-ranked cases (all selected by fixed rule)")
        fig.tight_layout()
        fig.savefig(OUTPUT_ROOT / f"ranked_examples_{safe_label}.png", dpi=160, bbox_inches="tight")
        plt.show()
        plt.close(fig)

pd.DataFrame(case_rows).to_csv(OUTPUT_ROOT / "ranked_case_selection.csv", index=False)
print("Wrote distribution plots and ranked-case atlases.")

## 15. Save run manifest

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest = {
    "experiment": "full released paired I-HAZE and NH-HAZE inference evaluation",
    "scope": "fixed-checkpoint inference only; no training, fine-tuning, coefficient selection, or per-image selection",
    "datasets": {
        name: {
            "root": str(DATASET_ROOTS[name]),
            "expected_all_release_pairs": EXPECTED_TOTALS[name],
            "observed_by_split": {item["split"]: item["n_pairs"] for item in LAYOUTS[name]},
            "all_release_contains_train_validation_test_if_present": True,
        }
        for name in LAYOUTS
    },
    "checkpoint_paths_sha256": {
        str(seed): {"path": str(path), "sha256": sha256_file(path)}
        for seed, path in CHECKPOINTS.items()
    },
    "model_settings": {"variant": "full", "blend": 0.08, "epsilon": 0.1, "weights_fixed": True},
    "input_protocol": {
        "preprocessing": "radiometric RGB [0,1]",
        "resolution": "native input dimensions",
        "large_image_inference": "project predict_full overlapping tiles; tile size 512, overlap 96",
        "color_pair_alignment": "image dimensions checked before inference",
    },
    "metrics": {
        "higher_is_better": ["psnr", "ssim", "ms_ssim"],
        "lower_is_better": ["mae", "deltaE2000", "lpips", "dists"],
        "perceptual_implementation": "IQA-PyTorch (pyiqa); LPIPS and DISTS; CPU",
    },
    "bootstrap": {
        "samples": BOOTSTRAP_SAMPLES, "seed": BOOTSTRAP_SEED,
        "unit": "image; retain the three fixed checkpoint outcomes",
        "interpretation": "conditional on the released images and fixed checkpoints",
    },
    "interpretation_limits": [
        "The official test split is reported separately when directory labels are present.",
        "Pooled all-release results include train/validation images and are descriptive, not official test scores.",
        "These results do not isolate component effects, establish architecture superiority, or validate deployment readiness.",
        "The five-image official test subsets remain small even when additional released images are summarized separately.",
    ],
    "software": {"python": platform.python_version(), "tensorflow": tf.__version__,
                 "torch": torch.__version__, "pyiqa": pyiqa.__version__, "opencv": cv2.__version__},
    "visible_tensorflow_gpus": [d.name for d in tf.config.list_physical_devices("GPU")],
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with (OUTPUT_ROOT / "run_manifest.json").open("w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print("Manifest saved:", OUTPUT_ROOT / "run_manifest.json")
print("Output files so far:", sum(p.is_file() for p in OUTPUT_ROOT.rglob("*")))

## 16. Export all task outputs (last cell)

Run this last after the evaluation, summaries, figures, and manifest finish. It recursively adds every regular output file and verifies the ZIP archive.

In [ ]:
import zipfile

ZIP_PATH = Path("/kaggle/working/full_release_real_haze_perceptual_eval.zip")
output_files = sorted(p for p in OUTPUT_ROOT.rglob("*") if p.is_file())
if not output_files:
    raise RuntimeError(f"No task outputs found under {OUTPUT_ROOT}")
expected_members = {p.relative_to(OUTPUT_ROOT).as_posix() for p in output_files}
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in output_files:
        archive.write(file_path, arcname=file_path.relative_to(OUTPUT_ROOT).as_posix())
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    archived_members = set(archive.namelist())
    damaged_member = archive.testzip()
if archived_members != expected_members:
    raise RuntimeError(
        f"ZIP member mismatch; missing={sorted(expected_members - archived_members)}, "
        f"unexpected={sorted(archived_members - expected_members)}"
    )
if damaged_member is not None:
    raise RuntimeError(f"ZIP integrity check failed at {damaged_member}")
print(f"Verified {len(expected_members)} output files in {ZIP_PATH}")
print(f"Archive size: {ZIP_PATH.stat().st_size / (1024 * 1024):.2f} MiB")